<a href="https://colab.research.google.com/github/MariethDataSc/ClassActivities_Mastery/blob/main/DL_Actividad2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Actividad 2: Diseño y Construcción de una Red Neuronal Artificial

**Dataset:** Breast Cancer Wisconsin (Diagnostic)  
**Asignatura:** Deep Learning — Unidad 2  
**Autora:** Ing. Madelayne Loor

## FASE 1: Selección y Preparación del Dataset

**Dataset seleccionado:** Breast Cancer Wisconsin (Diagnostic), disponible en `sklearn.datasets`. Es un dataset tabular real, ampliamente validado en Deep Learning (569 registros, 30 características numéricas derivadas de imágenes digitalizadas de biopsias).

In [2]:
# Importación de librerías
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

print("TensorFlow version:", tf.__version__)

# Semilla fija para la reproducibilidad en todos los experimentos
SEED = 42
tf.random.set_seed(SEED)

TensorFlow version: 2.20.0


In [3]:
# Carga del dataset
data = load_breast_cancer(as_frame=True)
X = data.data
y = data.target

print("Forma de X:", X.shape)
print("Forma de y:", y.shape)
print("\nClases:", dict(zip(data.target_names, [0, 1])))
print("\nDistribución de clases:")
print(y.value_counts())

X.head()

Forma de X: (569, 30)
Forma de y: (569,)

Clases: {np.str_('malignant'): 0, np.str_('benign'): 1}

Distribución de clases:
target
1    357
0    212
Name: count, dtype: int64


,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst radius,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


### Preprocesamiento: División 80/20 y Normalización

Se divide el dataset en 80% entrenamiento y 20% prueba (`test_size=0.2`), y se aplica **normalización (estandarización z-score)** con `StandardScaler`, ajustado únicamente sobre el conjunto de entrenamiento para evitar fuga de información hacia el conjunto de prueba.

In [4]:
# División 80% training / 20% testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

print("Training set:", X_train.shape, "-", round(len(X_train)/len(X)*100, 1), "%")
print("Testing set:", X_test.shape, "-", round(len(X_test)/len(X)*100, 1), "%")

# Normalización: se ajusta solo con datos de entrenamiento
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("\nMedia por característica tras escalar (train):", np.round(X_train_scaled.mean(axis=0)[:5], 3))
print("Desv. estándar por característica tras escalar (train):", np.round(X_train_scaled.std(axis=0)[:5], 3))

Training set: (455, 30) - 80.0 %
Testing set: (114, 30) - 20.0 %

Media por característica tras escalar (train): [-0.  0. -0. -0.  0.]
Desv. estándar por característica tras escalar (train): [1. 1. 1. 1. 1.]


## FASE 2: Justificación del Diseño Arquitectónico

### Arquitectura propuesta

| Capa | Neuronas | Activación | Justificación |
|---|---|---|---|
| Entrada | 30 | — | Corresponde a las 30 características del dataset |
| Oculta 1 | 16 | ReLU | Introduce no linealidad; evita el problema de desvanecimiento de gradiente propio de Sigmoide/Tanh en capas profundas |
| Oculta 2 | 8 | ReLU | Reduce progresivamente la dimensionalidad, forzando a la red a aprender representaciones más abstractas y compactas |
| Salida | 1 | Sigmoide | Problema de clasificación **binaria**: Sigmoide comprime la salida a un rango (0,1) interpretable como probabilidad de pertenecer a la clase positiva |

### 1. Funciones de Activación

- **Capas ocultas → ReLU:** se eligió ReLU (`f(z) = max(0, z)`) porque acelera la convergencia respecto a Sigmoide/Tanh y mitiga el desvanecimiento del gradiente, al no saturar para valores positivos. Es la opción estándar recomendada para capas ocultas en arquitecturas MLP.
- **Capa de salida → Sigmoide:** dado que el problema es de clasificación binaria (maligno/benigno), se requiere una salida interpretable como probabilidad en el rango (0,1). Sigmoide es la función correcta para este propósito (a diferencia de Softmax, que se usa en clasificación multiclase).

### 2. Función de Pérdida

Se utiliza **Binary Cross-Entropy** (`binary_crossentropy`):

$$L = -\left[y \ln(\hat{y}) + (1-y)\ln(1-\hat{y})\right]$$

Es la función de pérdida teóricamente correcta para clasificación binaria con salida Sigmoide, ya que penaliza fuertemente las predicciones confiadas pero incorrectas, y su gradiente combinado con Sigmoide produce actualizaciones de peso bien comportadas (evita el problema de gradientes muy pequeños que sí ocurre si se usara, por ejemplo, MSE con Sigmoide).

### Métrica de evaluación
Se usa **Accuracy** como métrica adicional de monitoreo (junto a la pérdida), dado que las clases están razonablemente balanceadas (~63% / ~37%).